In [ ]:
from transformers import BertForSequenceClassification, BertTokenizerFast

model_name = "bert-base-uncased"

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained(
    model_name
)

model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels = 2
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from datasets import load_dataset

# Using the namespace-qualified identifier to bypass the parser issue on cached older packages
imdb = load_dataset("stanfordnlp/imdb")

train_dataset = imdb["train"].shuffle(seed=42).select(range(5000))
test_dataset = imdb["test"].shuffle(seed=42).select(range(2000))

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [ ]:
trainig_args = TrainingArguments(
    output_dir = "mini-bert-mlm-nsp-model",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 5,
    weight_decay = 0.01,
    logging_steps = 100,
    fp16 = True
)

In [ ]:
trainer = Trainer(
    model = model,
    args = trainig_args,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    compute_metrics = compute_metrics
)

In [ ]:
trainer.train()

Step,Training Loss
100,0.523052
200,0.388282
300,0.377265
400,0.233044
500,0.241548
600,0.241872
700,0.159917
800,0.139264
900,0.144100
1000,0.100885


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1565, training_loss=0.18121990273935726, metrics={'train_runtime': 263.1193, 'train_samples_per_second': 95.014, 'train_steps_per_second': 5.948, 'total_flos': 1644444096000000.0, 'train_loss': 0.18121990273935726, 'epoch': 5.0})

In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy
0.057905,0.691320,1565,0.863000


{'eval_loss': 0.6913197040557861, 'eval_accuracy': 0.863}

In [ ]:
import torch
import torch.nn.functional as F

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
def predict(text):
  inputs = tokenizer(
      text,
      return_tensors = "pt",
      padding = True,
      truncation = True,
      max_length = 128
  )
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = model(**inputs)

  probs = F.softmax(outputs.logits, dim=1)
  prediction = torch.argmax(probs, dim=1).item()

  label = "Positive" if prediction == 1 else "Negative"
  return {
      "Text": text,
      "Prediction": label,
      "Probability": probs[0][prediction].item()
  }

In [ ]:
text = "This was the worst film I have ever seen in my life."
predict(text)

{'Text': 'This was the worst film I have ever seen in my life.',
 'Prediction': 'Negative',
 'Probability': 0.9989556074142456}

In [ ]:
text = "This was the amazing film I have ever seen in my life."
predict(text)

{'Text': 'This was the amazing film I have ever seen in my life.',
 'Prediction': 'Positive',
 'Probability': 0.9991507530212402}